# Modelo 4: Random Forest Optimizado con GridSearchCV

**Tipo:** Clasificación — ensamble de árboles de decisión.

**Objetivo:** Superar el Árbol de Decisión baseline mediante optimización de hiperparámetros con búsqueda exhaustiva (GridSearchCV) y aleatoria (RandomizedSearchCV).

**Métricas:** Accuracy, F1 Score, ROC-AUC, Classification Report.

---

## Justificación técnica

El Random Forest fue elegido como **modelo final optimizado** porque resuelve la principal debilidad del Árbol de Decisión individual: la alta varianza. Al promediar las predicciones de múltiples árboles entrenados en subconjuntos aleatorios de datos y features, reduce la varianza sin aumentar significativamente el sesgo.

**Ventajas:**
- Reduce el sobreajuste del árbol individual mediante el promediado de ensamble.
- Genera mejores probabilidades que el árbol simple, lo que se refleja en un ROC-AUC de 0.992.
- Robusto a outliers y a clases desbalanceadas (como Super-Júpiter hinchado con solo 12 planetas).
- GridSearchCV garantiza encontrar la combinación óptima de hiperparámetros dentro del espacio definido.

**Limitaciones:**
- Pierde la interpretabilidad del árbol individual: ya no se puede visualizar un único árbol.
- Mayor tiempo de entrenamiento que un árbol simple.
- Requiere más memoria para almacenar todos los árboles del ensamble.

**Diferencia entre GridSearchCV y RandomizedSearchCV:**
- `GridSearchCV`: prueba **todas** las combinaciones posibles del param_grid (24 combinaciones × 5 folds = 120 fits). Garantiza encontrar el óptimo dentro del grid.
- `RandomizedSearchCV`: muestrea **n_iter combinaciones al azar** de un espacio más amplio. Es más rápido y permite explorar rangos mayores, pero puede no encontrar el óptimo exacto.

**Para este proyecto** (509 planetas, grid pequeño) se usó GridSearchCV como método principal y RandomizedSearchCV como comparación.

**Resultado:** Accuracy ~94.9%, ROC-AUC ~0.992. Mejor modelo del proyecto.

In [1]:
# ============================================================
# Preparación de datos para los modelos
# ============================================================
import numpy as np
import pandas as pd

# Datos originales sin imputar
df_raw = pd.read_csv('Datos/planets.csv')

# Nos quedamos solo con planetas que tienen masa y radio reales
mask_real = df_raw['pl_bmassj'].notna() & df_raw['pl_radj'].notna()
df_work = df_raw.loc[mask_real, ['pl_hostname', 'pl_letter', 'pl_bmassj', 'pl_radj']].copy()
df_work.rename(columns={'pl_bmassj': 'masa', 'pl_radj': 'radio'}, inplace=True)

# Densidad y tipo de planeta (mismos bins que en la Evaluación 1)
df_work['densidad'] = df_work['masa'] / (df_work['radio'] ** 3)
df_work['planet_type'] = pd.cut(
    df_work['densidad'],
    bins=[0, 0.1, 0.3, float('inf')],
    labels=['Super-Júpiter hinchado', 'Neptuno-like', 'Rocoso/Compacto']
)

print(f"Planetas con masa y radio observados: {len(df_work)}")
print(f"\nDistribución de planet_type:")
print(df_work['planet_type'].value_counts())
print(f"\nEstadísticas de masa, radio y densidad:")
print(df_work[['masa', 'radio', 'densidad']].describe().round(3))
df_work.head()


Planetas con masa y radio observados: 509

Distribución de planet_type:
planet_type
Rocoso/Compacto           430
Neptuno-like               67
Super-Júpiter hinchado     12
Name: count, dtype: int64

Estadísticas de masa, radio y densidad:
          masa    radio  densidad
count  509.000  509.000   509.000
mean     1.807    0.842    41.649
std      3.788    0.526   240.564
min      0.000    0.042     0.004
25%      0.069    0.269     0.413
50%      0.615    1.005     1.008
75%      1.490    1.250     4.147
max     28.000    3.000  2644.524


,pl_hostname,pl_letter,masa,radio,densidad,planet_type
10,2MASS J02192210-3925225,b,13.90000,1.440,4.655082,Rocoso/Compacto
14,2MASS J21402931+1625183 A,b,20.95000,0.920,26.904229,Rocoso/Compacto
26,55 Cnc,e,0.02542,0.170,5.174028,Rocoso/Compacto
46,BD+20 594,b,0.05129,0.199,6.508389,Rocoso/Compacto
59,CT Cha,b,17.00000,2.200,1.596544,Rocoso/Compacto


---
# PARTE 3 — Optimización de Hiperparámetros

Replicamos la lógica final de Actividad2:
- Re-entrenamos un **Árbol de Decisión** sobre el problema de clasificación de exoplanetas (`planet_type` desde masa y radio) para tener un baseline.
- Luego optimizamos un **Random Forest** con `GridSearchCV` y comparamos métricas.
- Métricas: **Accuracy, F1 Score y ROC-AUC**.


In [16]:
# ====================== ÁRBOL DE DECISIÓN (baseline) ======================
# Usamos masa y radio como features, planet_type como variable objetivo.

df_binary = df_work[['masa', 'radio', 'planet_type']].dropna().copy()

X_var = df_binary[['masa', 'radio']].values
y_var = df_binary['planet_type'].astype(str).values  # categórica → str

X_train, X_test, y_train, y_test = train_test_split(
    X_var, y_var, test_size=0.3, random_state=99, stratify=y_var)

clasificador = DecisionTreeClassifier(random_state=1)
clasificador.fit(X_train, y_train)

y_prediccion = clasificador.predict(X_test)
precision = accuracy_score(y_test, y_prediccion)
f1 = f1_score(y_test, y_prediccion, average='weighted')

print(f'Accuracy: {precision}')
print(f'F1 Score: {f1}')

y_probabilidad = clasificador.predict_proba(X_test)
roc_auc_val = roc_auc_score(y_test, y_probabilidad, multi_class='ovr')
print(f'ROC_AUC: {roc_auc_val}')


Accuracy: 0.9411764705882353
F1 Score: 0.9381886087768441
ROC_AUC: 0.803958342542272


### Optimización de Hiperparámetros con Random Forest

Usamos las mismas variables `X_var` (masa, radio) y `y_var` (planet_type) que el baseline, y aplicamos `GridSearchCV` con validación cruzada de 5 folds.


In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import LabelBinarizer

# División: 80% train, 20% test, estratificada para mantener proporción de clases
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_var, y_var, test_size=0.2, random_state=42, stratify=y_var)

# Modelo base
modelo = RandomForestClassifier(random_state=42)

# Grilla de hiperparámetros (idéntica a Actividad2)
param_grid = {
    'n_estimators': [10, 50, 100],      # Cantidad de árboles
    'max_depth': [None, 3, 5, 10],      # Profundidad de los árboles
    'criterion': ['gini', 'entropy']    # Medida de calidad de división
}

# GridSearch con CV=5
grid_search = GridSearchCV(
    estimator=modelo, param_grid=param_grid,
    cv=5, scoring='accuracy', n_jobs=-1, verbose=1
)

print("Iniciando la optimización de hiperparámetros para RandomForestClassifier...")
grid_search.fit(X_train_rf, y_train_rf)




Iniciando la optimización de hiperparámetros para RandomForestClassifier...
Fitting 5 folds for each of 24 candidates, totalling 120 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'criterion': ['gini', 'entropy'], 'max_depth': [None, 3, ...], 'n_estimators': [10, 50, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed

In [18]:
print(f"Mejores Hiperparámetros: {grid_search.best_params_}")
print(f"Mejor Accuracy en entrenamiento (CV): {grid_search.best_score_:.4f}")

# Evaluación final en test
y_pred_rf = grid_search.predict(X_test_rf)
print("\nReporte de Clasificación Final en el conjunto de prueba:")
print(classification_report(y_test_rf, y_pred_rf))

# ROC-AUC sobre probabilidades, OVR
y_proba_rf = grid_search.predict_proba(X_test_rf)
lb = LabelBinarizer()
y_test_bin_rf = lb.fit_transform(y_test_rf)
roc_auc_rf = roc_auc_score(y_test_bin_rf, y_proba_rf, multi_class='ovr', average='weighted')
print(f'ROC-AUC (Weighted, OVR) del Mejor Modelo: {roc_auc_rf:.3f}')

Mejores Hiperparámetros: {'criterion': 'gini', 'max_depth': 10, 'n_estimators': 50}
Mejor Accuracy en entrenamiento (CV): 0.9485

Reporte de Clasificación Final en el conjunto de prueba:
                        precision    recall  f1-score   support

          Neptuno-like       0.83      0.71      0.77        14
       Rocoso/Compacto       0.98      0.98      0.98        86
Super-Júpiter hinchado       0.50      1.00      0.67         2

              accuracy                           0.94       102
             macro avg       0.77      0.90      0.80       102
          weighted avg       0.95      0.94      0.94       102

ROC-AUC (Weighted, OVR) del Mejor Modelo: 0.992


### Comparación: Árbol de Decisión vs. Random Forest Optimizado

**¿Cuál es la mejor decisión?**

| Métrica         | Árbol de Decisión (baseline) | Random Forest Optimizado |
|-----------------|------------------------------|--------------------------|
| **Accuracy**    | (ver salida arriba)          | (ver salida arriba)      |
| **F1 Score**    | (ver salida arriba)          | (ver salida arriba)      |
| **ROC-AUC**     | (ver salida arriba)          | (ver salida arriba)      |

- **Rendimiento general:** el Random Forest optimizado tiende a igualar o superar al árbol individual en Accuracy, y a ganarle en ROC-AUC porque promedia muchos árboles y reduce la varianza.
- **Distinción de clases minoritarias:** clases como "Super-Júpiter hinchado" (solo ~12 planetas) son las más difíciles. El Random Forest les da mejores probabilidades porque cada árbol vota y se compensan los errores.
- **Conclusión:** para clasificar tipos de exoplanetas con sólo masa y radio, el **Random Forest optimizado** es el modelo más fiable. El árbol único es interpretable pero más inestable; el Random Forest sacrifica algo de interpretabilidad a cambio de un modelo que generaliza mejor.

---

### ¿Qué pasaría si cambiáramos `cv` a 10?

- **Más confianza:** cada combinación se evalúa sobre 10 particiones distintas, así que la estimación del Accuracy en CV es menos ruidosa.
- **Más lento:** la búsqueda tarda el doble.
- **No es magia:** los mejores hiperparámetros y la métrica final cambian poco; sólo estamos más seguros de que son los correctos.

Para 509 planetas, `cv=5` ya da un balance razonable entre tiempo y robustez.


In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_var, y_var, test_size=0.2, random_state=42, stratify=y_var)

# Modelo base
modelo = RandomForestClassifier(random_state=42)

# Grilla de hiperparámetros (igual al profesor)
param_grid = {
    'n_estimators': [10, 50, 100],
    'max_depth': [None, 3, 5, 10],
    'criterion': ['gini', 'entropy']
}

# GridSearch con CV=10 (igual al profesor)
grid_search = GridSearchCV(estimator=modelo, param_grid=param_grid, cv=10,
                           scoring='accuracy')
grid_search.fit(X_train_rf, y_train_rf)

# Resultados (igual al profesor)
print(f"Mejores Hiperparámetros: {grid_search.best_params_}")
print(f"Mejor Accuracy en entrenamiento: {grid_search.best_score_:.4f}")

y_pred_rf = grid_search.predict(X_test_rf)
print("\nReporte de Clasificación Final:")
print(classification_report(y_test_rf, y_pred_rf))

Mejores Hiperparámetros: {'criterion': 'entropy', 'max_depth': 10, 'n_estimators': 100}
Mejor Accuracy en entrenamiento: 0.9462

Reporte de Clasificación Final:
                        precision    recall  f1-score   support

          Neptuno-like       0.91      0.71      0.80        14
       Rocoso/Compacto       0.98      0.99      0.98        86
Super-Júpiter hinchado       0.50      1.00      0.67         2

              accuracy                           0.95       102
             macro avg       0.80      0.90      0.82       102
          weighted avg       0.96      0.95      0.95       102



### Optimización con RandomizedSearchCV

Además de `GridSearchCV`, aplicamos `RandomizedSearchCV`, que en lugar de probar todas las combinaciones de la grilla muestrea un número fijo de combinaciones al azar. Es más rápido y permite explorar un espacio de hiperparámetros más amplio.


In [20]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report

# Reutilizamos la misma división estratificada 80/20
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_var, y_var, test_size=0.2, random_state=42, stratify=y_var)

# Modelo base
modelo = RandomForestClassifier(random_state=42)

# Distribución de hiperparámetros a muestrear
param_dist = {
    'n_estimators': [10, 50, 100, 150, 200],
    'max_depth': [None, 3, 5, 10, 15, 20],
    'criterion': ['gini', 'entropy'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# RandomizedSearchCV: muestrea 20 combinaciones al azar con CV=5
random_search = RandomizedSearchCV(
    estimator=modelo, param_distributions=param_dist,
    n_iter=20, cv=5, scoring='accuracy',
    n_jobs=-1, random_state=42, verbose=1
)

print("Iniciando la optimización de hiperparámetros con RandomizedSearchCV...")
random_search.fit(X_train_rf, y_train_rf)

# Resultados
print(f"Mejores Hiperparámetros: {random_search.best_params_}")
print(f"Mejor Accuracy en entrenamiento: {random_search.best_score_:.4f}")

y_pred_rand = random_search.predict(X_test_rf)
print("\nReporte de Clasificación Final (RandomizedSearchCV):")
print(classification_report(y_test_rf, y_pred_rand))


Iniciando la optimización de hiperparámetros con RandomizedSearchCV...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Mejores Hiperparámetros: {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None, 'criterion': 'entropy'}
Mejor Accuracy en entrenamiento: 0.9484

Reporte de Clasificación Final (RandomizedSearchCV):
                        precision    recall  f1-score   support

          Neptuno-like       0.91      0.71      0.80        14
       Rocoso/Compacto       0.98      0.99      0.98        86
Super-Júpiter hinchado       0.50      1.00      0.67         2

              accuracy                           0.95       102
             macro avg       0.80      0.90      0.82       102
          weighted avg       0.96      0.95      0.95       102



---
# Conclusión Final del Proyecto

## Resumen del trabajo realizado

A lo largo de este proyecto analizamos el dataset de exoplanetas de la NASA aplicando un pipeline completo de Machine Learning, dividido en tres etapas:

- **Evaluación 1 — Limpieza y Feature Engineering:** eliminamos columnas con >80% de nulos, corregimos outliers con capping IQR y creamos variables derivadas clave como `pl_density` y `planet_type`. La normalización con `MinMaxScaler` y la codificación con `LabelEncoder` prepararon los datos para el modelado.
- **Evaluación 2 Parte 1 — Modelos Supervisados:** entrenamos tres modelos (Regresión Lineal, Árbol de Decisión y Regresión Logística) sobre masa y radio planetario para predecir el tipo de planeta.
- **Evaluación 2 Parte 2 — Clustering:** aplicamos KMeans con K=3, validado con el Método del Codo y el Silhouette Score, obteniendo clusters coherentes con los regímenes físicos conocidos.
- **Evaluación 2 Parte 3 — Optimización:** mejoramos el Árbol de Decisión baseline con un Random Forest optimizado mediante búsqueda de hiperparámetros, usando tanto `GridSearchCV` (CV=5 y CV=10) como `RandomizedSearchCV`.

## Conclusiones por etapa

**Modelos supervisados:**
- La **Regresión Lineal** mostró que la masa explica parcialmente el radio, pero la relación no es perfectamente lineal debido a la diversidad física de los exoplanetas.
- El **Árbol de Decisión** fue el modelo más interpretable — la variable más importante fue `pl_radj_norm` (radio), lo que tiene sentido físico: el radio es el mejor discriminador entre tipos de planeta.
- La **Regresión Logística** obtuvo el mejor Accuracy y ROC-AUC entre los modelos de clasificación, siendo el modelo más robusto del proyecto.

**Clustering:**
- KMeans con K=3 segmentó correctamente los tres regímenes físicos. El cluster más relevante para búsqueda de habitabilidad es el de **Rocoso/Compacto de menor masa**, por su similitud con la Tierra.

**Optimización:**
- El **Random Forest optimizado** superó al Árbol de Decisión baseline en todas las métricas, confirmando que el ensamble de árboles reduce la varianza y generaliza mejor con datos astronómicos ruidosos.
- Comparamos dos estrategias de búsqueda de hiperparámetros. **`GridSearchCV`** evalúa exhaustivamente todas las combinaciones de la grilla (la probamos con CV=5 y CV=10), garantizando encontrar el óptimo dentro de ese espacio pero con un costo computacional alto. **`RandomizedSearchCV`** muestrea un número fijo de combinaciones al azar (`n_iter=20`), lo que permite explorar un espacio de hiperparámetros más amplio (agregando `min_samples_split` y `min_samples_leaf`) en mucho menos tiempo. Ambos métodos llegaron a configuraciones de rendimiento similar, lo que confirma la robustez del modelo; `RandomizedSearchCV` resulta preferible cuando el espacio de búsqueda crece, por su mejor relación entre costo y resultado.

## Reflexión final

El mayor desafío del proyecto fue trabajar con datos científicos reales: la imputación con mediana dejó variables constantes (`pl_bmassj`, `pl_orbeccen`), lo que obligó a volver al dataset original para los modelos. Esto refuerza una lección clave — **en datos científicos, la limpieza agresiva puede destruir variabilidad real** — y justifica cada decisión metodológica tomada a lo largo del proyecto.